In [2]:
import apache_beam as beam
from apache_beam.metrics import Metrics
import logging
import json

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("RetailSalesPipeline")

class ValidateAndTransform(beam.DoFn):
    def __init__(self):
        # Metrics to monitor data quality
        self.invalid_records = Metrics.counter(self.__class__, 'invalid_records')
        self.valid_records = Metrics.counter(self.__class__, 'valid_records')

    def process(self, element):
        try:
            # Split the CSV line into fields
            fields = element.split(",")
            order_id = fields[0]
            product_id = fields[1]
            category = fields[2]
            quantity = int(fields[3])
            price = float(fields[4])

            # Validation checks
            if not order_id or not product_id or not category:
                raise ValueError("Missing required fields")

            if quantity <= 0:
                raise ValueError(f"Invalid quantity: {quantity}")

            if price <= 0:
                raise ValueError(f"Invalid price: {price}")

            # Calculate total sale amount
            total_sale = quantity * price
            self.valid_records.inc()
            
            # Return the transformed data
            yield {
                'order_id': order_id,
                'product_id': product_id,
                'category': category,
                'quantity': quantity,
                'price': price,
                'total_sale': total_sale
            }

        except Exception as e:
            # Log the error and increment the invalid record counter
            logger.error(f"Error processing record {element}: {e}")
            self.invalid_records.inc()

# Aggregator to calculate total sales per category
class SumSalesPerCategory(beam.DoFn):
    def process(self, element):
        category, sales = element
        total_sales = sum(sales)
        yield (category, total_sales)

# Define the pipeline options (e.g., GCP project, runner)
from apache_beam.options.pipeline_options import PipelineOptions

options = PipelineOptions(
    project='your-gcp-project',
    runner='DataflowRunner',  # Replace with 'DirectRunner' for local testing
    temp_location='gs://dummy-bucket-gcs-test/temp'
)

# Define the pipeline
with beam.Pipeline() as pipeline:
    sales_data = (
        pipeline
        | "Read Sales Data" >> beam.io.ReadFromText("sales_data.csv", skip_header_lines=1)
        | "Validate and Transform" >> beam.ParDo(ValidateAndTransform())
        | "Extract Category and Total Sale" >> beam.Map(lambda record: (record['category'], record['total_sale']))
        | "Group Sales By Category" >> beam.GroupByKey()
        | "Sum Sales Per Category" >> beam.ParDo(SumSalesPerCategory())
        | "Write Output" >> beam.io.WriteToText("gs://dummy-bucket-gcs-test/sales_output", file_name_suffix=".json")
    )

    # Use Metrics to monitor pipeline execution
    result = pipeline.run()
    result.wait_until_finish()

    # Get metrics after pipeline completion
    try:
        metrics = result.metrics()
        invalid_count = metrics.query()['counters'][0].result
        valid_count = metrics.query()['counters'][1].result

        logger.info(f"Total valid records processed: {valid_count}")
        logger.info(f"Total invalid records encountered: {invalid_count}")

    except Exception as e:
        logger.error(f"Error retrieving metrics: {e}")


INFO:root:Missing pipeline option (runner). Executing pipeline using the default runner: DirectRunner.
INFO:apache_beam.runners.worker.statecache:Creating state cache with size 104857600
ERROR:RetailSalesPipeline:Error processing record 1005,P005,Clothing,-2,19.99  # Invalid quantity: could not convert string to float: '19.99  # Invalid quantity'
ERROR:RetailSalesPipeline:Error processing record 1006,P006,Electronics,1,-99.99  # Invalid price: could not convert string to float: '-99.99  # Invalid price'
ERROR:RetailSalesPipeline:Error processing record 1007,,Toys,2,49.99  # Missing product_id: could not convert string to float: '49.99  # Missing product_id'
INFO:apache_beam.io.gcp.gcsio:Finished listing 1 files in 0.2586359977722168 seconds.
INFO:apache_beam.io.filebasedsink:Starting finalize_write threads with num_shards: 1 (skipped: 0), batches: 1, num_threads: 1
INFO:apache_beam.io.filebasedsink:Renamed 1 shards in 0.84 seconds.
INFO:RetailSalesPipeline:Total valid records processed